# Infinite-Horizon CRRA Benchmark

Standalone orchestration notebook for the stationary no-income, no-mortality benchmark in `inf_horizon_solver.py`.

This notebook is designed to stay separate from `main.ipynb` and the lifecycle notebooks. The typical workflow is:

1. build the same annual VAR-based model and precompute object
2. optionally run a nested-JIT smoke test
3. run the infinite-horizon benchmark solver from the built-in Markowitz cold-start
4. inspect diagnostics and convergence histories

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from inf_horizon_solver import (
    compile_inner_kernel_smoke_test,
    run_infinite_horizon_solver,
)
from model import DiscretizationConfig, SolverConfig
from precompute import Precompute, build_model
from var import (
    build_nominal_system1_var_config,
    build_nominal_system1_var_config_hardcoded,
)

## Configuration Surface

The notebook exposes four configuration layers:

1. notebook flow flags: whether to use the hardcoded VAR, run the smoke test, and run the cold-start cross-check
2. `BASE_CONFIG`: economic calibration passed into `build_model(...)`
3. `DISC_CONFIG`: discretization choices via `DiscretizationConfig(...)`
4. `IH_SOLVE_OPTIONS`: runtime options passed directly into `run_infinite_horizon_solver(...)`

The notebook now exposes two discretization presets:

- `"small"`: cheap smoke-test grid for debugging the workflow
- `"main_like"`: close to the current `main.ipynb` production retirement loadout and better for a benchmark you want to trust

For a real benchmark figure, `"main_like"` is the recommended choice. The small profile is mainly for quick debugging.

The current defaults are set for a **main-like benchmark run**: use the main-like discretization, solve the model in unconstrained mode, and initialize from the built-in Markowitz cold-start.

In [ ]:
USE_HARDCODED_VAR = False
RUN_SMOKE_TEST = False
RUN_COLD_START_CHECK = False
DISC_CONFIG_PROFILE = "main_like"   # "small" | "main_like"

BASE_CONFIG = {
    "beta": 0.96,
    "gamma": 3.0,
    "b_bar": 10,
    "start_age": 22,
    "retire_age": 67,
    "terminal_age": 99,
    "b0": -6.142,
    "b1": 0.3040,
    "b2": -0.051,
    "b3": 0.002586,
    "rho": 0.991,
    "pz": 0.176,
    "mu_eta1": -0.524,
    "sigma_eta1": 0.113,
    "mu_eta2": -(0.176 / (1.0 - 0.176)) * (-0.524),
    "sigma_eta2": 0.046,
    "pe": 0.044,
    "mu_eps1": 0.134,
    "sigma_eps1": 0.762,
    "mu_eps2": 0.0,
    "sigma_eps2": 0.055,
    "constrained": True,
}

DISC_CONFIG_SMALL = DiscretizationConfig(
    n_wealth=10,
    n_savings=10,
    wealth_max=60.0,
    savings_max=60.0,
    state_grid_sizes=(3, 3, 3),
    state_grid_mode="principal",
    state_n_stds=2.0,
    n_z=2,
    n_stds=2.0,
    n_eps_nodes=2,
    n_eta_nodes=2,
    n_ret_nodes_1d=2,
    n_state_quad_nodes=2,
)

# Close to the current constrained principal-grid setup in main.ipynb,
# but using the retirement-focused quadrature loadout already used there for
# retirement-only work.
DISC_CONFIG_MAINLIKE = DiscretizationConfig(
    n_wealth=150,
    n_savings=150,
    wealth_max=200.0,
    savings_max=None,
    state_grid_sizes=(5, 5, 5),
    state_grid_mode="principal",
    state_n_stds=(0.6, 1.75, 2.0),
    n_z=2,
    n_stds=3.0,
    n_eps_nodes=3,
    n_eta_nodes=3,
    n_ret_nodes_1d=(3, 7, 5),
    n_state_quad_nodes=(2, 2, 5),
)

if DISC_CONFIG_PROFILE == "small":
    DISC_CONFIG = DISC_CONFIG_SMALL
elif DISC_CONFIG_PROFILE == "main_like":
    DISC_CONFIG = DISC_CONFIG_MAINLIKE
else:
    raise ValueError("DISC_CONFIG_PROFILE must be 'small' or 'main_like'")

SOLVER_CONFIG = SolverConfig()

IH_SOLVE_OPTIONS = {
    "tol": 1e-6,
    "max_iter": 400,
    "damping": 1.0,
    "trim_wealth_points": 2,
    "constrained": False,
    "run_smoke_test": False,
    "show_progress": True,
    "progress_every": 1,
    "progress_probe_wealth": 8.0,
    "verbose": True,
}

In [ ]:
csv_path = Path("data/var_dataset.csv")

if USE_HARDCODED_VAR or not csv_path.exists():
    var_config = build_nominal_system1_var_config_hardcoded()
    var_data = None
else:
    var_config, _, var_data = build_nominal_system1_var_config(csv_path=str(csv_path))

model = build_model(BASE_CONFIG, var_config, verbose=False)
pc = Precompute(model, DISC_CONFIG, verbose=False)

print(f"N_state = {pc.N_state}")
print(f"n_z = {pc.n_z}, n_w = {pc.n_w}, n_s = {pc.n_s}")
print(f"wealth grid: [{pc.wealth_grid[0]:.3e}, {pc.wealth_grid[-1]:.3e}]")

## Optional nested-JIT smoke test

Leave this off unless you explicitly want to test nested callability of the outer JIT core. The real benchmark solve below exercises the actual path you care about.

In [ ]:
if RUN_SMOKE_TEST:
    smoke = compile_inner_kernel_smoke_test(
        model,
        pc,
        solver_config=SOLVER_CONFIG,
        verbose=True,
    )
    smoke
else:
    print("RUN_SMOKE_TEST = False")

## Initializer

The main benchmark run below uses the built-in Markowitz cold-start from `inf_horizon_solver.py`. The solve itself is currently set to unconstrained mode via `IH_SOLVE_OPTIONS["constrained"] = False`.

In [ ]:
initializer_info = {
    "solver_mode": "unconstrained" if IH_SOLVE_OPTIONS["constrained"] is False else "model default",
    "initializer": "markowitz_cold_start",
}
initializer_info

## Infinite-horizon benchmark solve

In [ ]:
C_inf, S_inf, B_inf, ih_diag = run_infinite_horizon_solver(
    model,
    pc,
    solver_config=SOLVER_CONFIG,
    **IH_SOLVE_OPTIONS,
)

ih_summary = {
    "converged": ih_diag["converged"],
    "n_iter": ih_diag["n_iter"],
    "final_stopping_supnorm": ih_diag["final_stopping_supnorm"],
    "final_policy_supnorm": ih_diag["final_policy_supnorm"],
    "final_xi_supnorm": ih_diag["final_xi_supnorm"],
    "final_share_supnorm": ih_diag["final_share_supnorm"],
    "max_z_slice_diff_c": ih_diag["max_z_slice_diff_c"],
    "max_z_slice_diff_s": ih_diag["max_z_slice_diff_s"],
    "max_z_slice_diff_b": ih_diag["max_z_slice_diff_b"],
    "max_xi_spread_across_w": ih_diag["max_xi_spread_across_w"],
    "max_share_spread_across_w": ih_diag["max_share_spread_across_w"],
    "stability_proxy": ih_diag["stability_proxy"],
    "used_warm_start": ih_diag["used_warm_start"],
}
ih_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(ih_diag["xi_supnorm_history"], label="xi supnorm")
axes[0].semilogy(ih_diag["share_supnorm_history"], label="share supnorm")
axes[0].axhline(IH_SOLVE_OPTIONS["tol"], color="black", linestyle=":", label="tol")
axes[0].set_title("Stopping Metrics")
axes[0].set_xlabel("Iteration")
axes[0].legend()

axes[1].semilogy(ih_diag["policy_supnorm_history"], color="tab:purple")
axes[1].set_title("Absolute Policy Supnorm")
axes[1].set_xlabel("Iteration")

plt.tight_layout()

## Optional Rerun

The main run already uses the Markowitz cold-start. This section is left only as a place to rerun the benchmark if you want to tweak options and compare outputs manually.

In [ ]:
if RUN_COLD_START_CHECK:
    C_cold, S_cold, B_cold, cold_diag = run_infinite_horizon_solver(
        model,
        pc,
        solver_config=SOLVER_CONFIG,
        **IH_SOLVE_OPTIONS,
    )

    cold_compare = {
        "cold_converged": cold_diag["converged"],
        "cold_n_iter": cold_diag["n_iter"],
        "cold_final_stopping_supnorm": cold_diag["final_stopping_supnorm"],
        "max_abs_diff_C": float(np.max(np.abs(C_inf - C_cold))),
        "max_abs_diff_S": float(np.max(np.abs(S_inf - S_cold))),
        "max_abs_diff_B": float(np.max(np.abs(B_inf - B_cold))),
    }
    cold_compare
else:
    print("RUN_COLD_START_CHECK = False")

## Practical reading of the diagnostics

For a trustworthy benchmark run, the main objects to watch are:

- `converged`
- `final_stopping_supnorm`
- `max_z_slice_diff_c`, `max_z_slice_diff_s`, `max_z_slice_diff_b`
- `max_xi_spread_across_w`
- `max_share_spread_across_w`
- `stability_proxy`

Since the main run already uses the Markowitz initializer, the key things to watch are convergence quality, Newton failures, and whether the central-state allocation looks economically sensible.